In [6]:
import re
import time
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import json
import subprocess
from playwright_stealth import Stealth

In [ ]:
# RUN THIS IN TERMINAL TO LAUNCH CHROME WITH REMOTE DEBUGGING
# /Applications/Google\ Chrome.app/Contents/MacOS/Google\ Chrome --remote-debugging-port=9222 --user-data-dir="$HOME/tmp/chrome-debug"



DevTools listening on ws://127.0.0.1:9222/devtools/browser/7fd425b1-74ed-41b2-a7e3-9ac167b3db40
[35351:19444932:0414/205647.098260:ERROR:google_apis/gcm/engine/registration_request.cc:291] Registration response error message: DEPRECATED_ENDPOINT
[35351:19444869:0414/205648.898656:ERROR:chrome/browser/ui/views/user_education/impl/browser_user_education_interface_impl.cc:178] Attempting to show IPH IPH_BatterySaverMode before browser initialization complete; IPH will not be shown.
Trying to load the allocator multiple times. This is *not* supported.
[35351:19444869:0414/205649.057061:ERROR:chrome/browser/ui/webui/ntp/new_tab_ui.cc:53] Requested load of chrome://newtab/ for incorrect profile type.
[35351:19444869:0414/205651.341711:ERROR:chrome/browser/ui/webui/ntp/new_tab_ui.cc:53] Requested load of chrome://newtab/ for incorrect profile type.
[35351:19444869:0414/205656.084689:ERROR:chrome/browser/ui/webui/ntp/new_tab_ui.cc:53] Requested load of chrome://newtab/ for incorrect profile t

KeyboardInterrupt: 

In [7]:
async def main():
    async with Stealth().use_async(async_playwright()) as p:
        browser = await p.chromium.connect_over_cdp("http://localhost:9222")
        default_context = browser.contexts[0]
        page = default_context.pages[0]
        await page.goto("https://truthsocial.com/search")
        time.sleep(3.2)  # Wait for the page to load completely

        # Use if we need to log in, some auth sessions are cached and may not require this
        #await page.locator("button:text('Sign In')").click()
        #time.sleep(1.5)  # Wait for the sign-in modal to appear

        #await page.locator("input[placeholder='Username']").fill("BigPigeon27")
        #time.sleep(0.5)  # Wait for the password field to become active
        #await page.locator("input[placeholder='Password']").fill("amogusSUS!!")
        #time.sleep(0.5)  # Wait for the login button to become active
        #await page.locator("button:text('Sign In')").click()
        #time.sleep(3.5)  # Wait for the login process to complete


        await page.locator("button:text('Topics')").click()
        time.sleep(0.7)  # Wait for the topics to load

        #scroll down the page 35 px to load more topics
        #Truth uses agressive culling to only load a few topics at a time, this forces it to load more
        await page.evaluate("window.scrollBy(0, 90);")

        #find the div with id search-results
        search_results = page.locator("div#search-results")

        with open("truthsocial.html", "w") as f:
            f.write(await search_results.inner_html())

        await browser.close()

await main()

In [8]:
with open("truthsocial.html", "r") as f_html:
    soup = BeautifulSoup(f_html, "html.parser")

topic_links = soup.find_all("a")
for link in topic_links:
    print(link['href'].replace('/tags/', '', 1))

num_posts = soup.find_all("span")
numbers = []
for post in num_posts:
    if post.text.isnumeric() or re.match(r'^\d+(\.\d+)?[KkMm]$', post.text):
        numbers.append(post.text)
        print(post.text)

lines = soup.find_all("svg")
for line in lines:
    print(line)

outputJson = {"trends": []}
for i in range(len(topic_links)):
    topic = topic_links[i]['href'].replace('/tags/', '', 1)
    posts = numbers[i] if i < len(numbers) else "0"
    svg = str(lines[i]) if i < len(lines) else ""
    if svg.startswith("<svg "):
        svg = svg.replace("<svg preserveaspectratio=\"none\"", "<svg xmlns=\"http://www.w3.org/2000/svg\" ", 1)
        svg = svg.replace("\n", " ")
        #remove uneeded spaces
        svg = re.sub(r'\s+', ' ', svg).strip()
    outputJson["trends"].append({"topic": topic, "posts": posts, "svg": svg})

with open("truth.json", "w") as f_json:
    json.dump(outputJson, f_json, indent=4)

WETHEPEOPLE
MAGA
Trump
Truth
Iran
politics
XTeam
PassTheSaveAmericaAct
fraud
MarcoRubio
SaveAmerica
AmericaFirst
SaveOurChildren
1.26k
11k
7.49k
6.52k
3.78k
316
3.96k
1.19k
276
86
1.2k
4.39k
131
<svg preserveaspectratio="none" viewbox="0 0 40 28"><g><path d="M2 2 C 3.5 2 6.5 12.559999999999999 8 12.559999999999999 C 9.5 12.559999999999999 12.5 13.52 14 13.52 C 15.5 13.52 18.5 15.44 20 15.44 C 21.5 15.44 24.5 14.48 26 14.48 C 27.5 14.48 30.5 26 32 26 C 33.5 26 36.5 2 38 2 L38 26 2 26 2 2" style="stroke: none; stroke-width: 0; fill-opacity: 0.1; fill: none;"></path><path d="M2 2 C 3.5 2 6.5 12.559999999999999 8 12.559999999999999 C 9.5 12.559999999999999 12.5 13.52 14 13.52 C 15.5 13.52 18.5 15.44 20 15.44 C 21.5 15.44 24.5 14.48 26 14.48 C 27.5 14.48 30.5 26 32 26 C 33.5 26 36.5 2 38 2" style="stroke: rgb(129, 140, 248); stroke-width: 1; stroke-linejoin: round; stroke-linecap: round; fill: none;"></path></g></svg>
<svg preserveaspectratio="none" viewbox="0 0 40 28"><g><path d="M2 3.0714